# LeetCode 1006 — Clumsy Factorial bằng Python

Link đề: https://leetcode.com/problems/clumsy-factorial/ · Độ khó: **Medium** · Chủ đề: Math (toán), Stack (ngăn xếp), Simulation (mô phỏng)

Skeleton LeetCode:

```python
class Solution:
    def clumsy(self, n: int) -> int:
        ...
```

Ghi chú: **giai thừa vụng về** `clumsy(n)` dùng các số `n, n-1, ..., 1` theo thứ tự giảm dần, thay các phép nhân bằng một vòng lặp cố định các phép toán `*`, `/`, `+`, `-` (theo thứ tự đó), ví dụ `clumsy(10) = 10 * 9 / 8 + 7 - 6 * 5 / 4 + 3 - 2 * 1`. Các phép toán vẫn theo **thứ tự ưu tiên** thông thường (nhân/chia trước, cộng/trừ sau; nhân và chia xử lý từ trái sang phải) và phép chia là **chia lấy phần nguyên** (`10 * 9 / 8 = 90 / 8 = 11`). Tính `clumsy(n)`. Bài này luyện **mô phỏng biểu thức bằng ngăn xếp**, và có một bẫy lớn: **phép chia nguyên với số âm** (chia lấy sàn khác chia cắt bỏ phần thập phân). Ngoài ra có công thức `O(1)` nhờ tính tuần hoàn chu kỳ 4.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: biểu thức có thứ tự ưu tiên phép toán, chia nguyên, vòng lặp phép toán `* / + -`.
2. Nắm các công cụ Python cần thiết: ngăn xếp bằng `list`, `//` (chia **sàn**) và vì sao khác chia **cắt bỏ phần thập phân** khi có số âm, `divmod`, `n % 4`, `eval` trên chuỗi nhỏ để làm oracle, và `re.split`.
3. Hiểu nguyên lý **tính biểu thức bằng ngăn xếp**: nhân/chia thực hiện ngay trên phần tử đỉnh ngăn xếp; cộng/trừ đẩy một số (có dấu) vào ngăn xếp; kết quả là tổng ngăn xếp.
4. Đi qua 4 cách: dựng chuỗi biểu thức rồi tách thành các số hạng và tính (oracle), ngăn xếp với phép chia cắt bỏ phần thập phân (cách nộp chính), duyệt theo nhóm bốn số `a*b/c + d` (`O(n)` thời gian, `O(1)` bộ nhớ) và công thức `O(1)` theo `n mod 4`.
5. Tránh các lỗi kinh điển: dùng `//` trên **số âm** (cho `-8` thay vì `-7`), dùng `/` ra số thực, bỏ qua thứ tự ưu tiên (tính từ trái sang phải), lệch chu kỳ phép toán, và áp công thức `O(1)` cho `n <= 4`.

## 2. Đề bài & ví dụ

**Giai thừa** của số nguyên dương `n` là tích mọi số nguyên dương nhỏ hơn hoặc bằng `n`, ví dụ `factorial(10) = 10 * 9 * 8 * 7 * 6 * 5 * 4 * 3 * 2 * 1`.

Ta tạo **giai thừa vụng về** bằng cách dùng các số nguyên theo thứ tự giảm dần và thay các phép nhân bằng một vòng lặp cố định gồm phép nhân `'*'`, phép chia `'/'`, phép cộng `'+'` và phép trừ `'-'` theo thứ tự đó. Ví dụ `clumsy(10) = 10 * 9 / 8 + 7 - 6 * 5 / 4 + 3 - 2 * 1`.

Tuy nhiên các phép toán vẫn được áp dụng theo **thứ tự ưu tiên** quen thuộc: làm mọi phép nhân và chia trước, rồi mới đến cộng và trừ; nhân và chia xử lý từ trái sang phải. Phép chia là **chia lấy phần nguyên** (floor) sao cho `10 * 9 / 8 = 90 / 8 = 11`.

Cho số nguyên `n`, trả về giai thừa vụng về của `n`.

**Ví dụ 1:** `n = 4` -> `7` (`7 = 4 * 3 / 2 + 1`).

**Ví dụ 2:** `n = 10` -> `12` (`12 = 10 * 9 / 8 + 7 - 6 * 5 / 4 + 3 - 2 * 1`).

**Ràng buộc:**

- `1 <= n <= 10^4`

In [1]:
from typing import List
import re
import random, time

def build_expression(n: int) -> str:
    """Biểu thức của clumsy(n) viết bằng cú pháp Python (// là chia lấy phần nguyên)."""
    ops = ["*", "//", "+", "-"]
    parts = [str(n)]
    for i, num in enumerate(range(n - 1, 0, -1)):
        parts.append(ops[i % 4])
        parts.append(str(num))
    return "".join(parts)

examples = [(4, 7), (10, 12)]
for n, expected in examples:
    print(f"n={n} -> biểu thức {build_expression(n)} | kỳ vọng {expected}")

n=4 -> biểu thức 4*3//2+1 | kỳ vọng 7
n=10 -> biểu thức 10*9//8+7-6*5//4+3-2*1 | kỳ vọng 12


## 3. Kiến thức nền

### 3.1. Thứ tự ưu tiên và vòng lặp phép toán

Sau số `n` đến phép toán thứ nhất là `*`, rồi `/`, `+`, `-`, lại `*`, ... Các số `n, n-1, ..., 1` được nối bằng `n - 1` phép toán. Các phép toán `*` và `/` ưu tiên cao hơn nên **dính** các số bên cạnh thành các **số hạng**; sau đó các số hạng được cộng hoặc trừ.

In [2]:
print(build_expression(10))
print("4 phép toán đầu của n = 10:", ["*", "//", "+", "-"], "-> nhóm: 10*9//8 | +7 | -6*5//4 | +3 | -2*1")

10*9//8+7-6*5//4+3-2*1
4 phép toán đầu của n = 10: ['*', '//', '+', '-'] -> nhóm: 10*9//8 | +7 | -6*5//4 | +3 | -2*1


### 3.2. Chia sàn `//` và chia cắt bỏ phần thập phân với số âm

Python `//` là **chia sàn** (làm tròn về `-vô cực`): `-30 // 4 = -8` (vì `-7.5` làm tròn xuống thành `-8`). Nhưng "trừ `6 * 5 / 4`" nghĩa là `-(30 // 4) = -7`. Hai kết quả **khác nhau** khi có số âm. Chia **cắt bỏ phần thập phân** (truncation, như C/Java) cho `-30 / 4 = -7`. Với số dương hai cách trùng nhau.

In [3]:
print("-30 // 4 =", -30 // 4, "| -(30 // 4) =", -(30 // 4), "| int(-30 / 4) =", int(-30 / 4))

def tdiv(a: int, b: int) -> int:
    """Chia cắt bỏ phần thập phân (về 0) cho số nguyên, b > 0."""
    return -((-a) // b) if a < 0 else a // b

print("tdiv(-30, 4) =", tdiv(-30, 4), "| tdiv(30, 4) =", tdiv(30, 4), "| tdiv(-8, 4) =", tdiv(-8, 4))
assert tdiv(-30, 4) == -7 and tdiv(30, 4) == 7

-30 // 4 = -8 | -(30 // 4) = -7 | int(-30 / 4) = -7
tdiv(-30, 4) = -7 | tdiv(30, 4) = 7 | tdiv(-8, 4) = -2


### 3.3. Tính biểu thức bằng ngăn xếp

Ý tưởng chung cho bài "máy tính cơ bản": duyệt các số theo thứ tự; giữ một ngăn xếp các **số hạng** (đã có dấu). Phép `+`/`-` bắt đầu số hạng mới (đẩy `+x` hoặc `-x`); phép `*`/`/` biến đổi **số hạng đang dở** (đỉnh ngăn xếp). Cuối cùng cộng mọi số hạng trong ngăn xếp.

In [4]:
stack = [10]
stack[-1] *= 9                 # 10 * 9 = 90
stack[-1] //= 8                # 90 // 8 = 11
stack.append(7)                # + 7
stack.append(-6)               # - 6  (đẩy -6)
stack[-1] *= 5                 # -6 * 5 = -30
print("đỉnh sau nhân:", stack[-1], "| sàn:", stack[-1] // 4, "| cắt bỏ:", tdiv(stack[-1], 4))

đỉnh sau nhân: -30 | sàn: -8 | cắt bỏ: -7


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 2: `10 * 9 / 8 + 7 - 6 * 5 / 4 + 3 - 2 * 1`.

- Số hạng 1: `10 * 9 / 8 = 90 / 8 = 11`.
- Số hạng 2: `+ 7`.
- Số hạng 3: `- (6 * 5 / 4) = - (30 / 4) = - 7`.
- Số hạng 4: `+ 3`.
- Số hạng 5: `- (2 * 1) = - 2`.

Tổng: `11 + 7 - 7 + 3 - 2 = 12`.

Ví dụ 1 (`n = 4`): `4 * 3 / 2 + 1 = 12 / 2 + 1 = 6 + 1 = 7`.

Nhận xét: biểu thức chia thành các **nhóm bốn số** `a * b / c + d`, trong đó `a, b, c, d` là bốn số liên tiếp giảm dần (nhóm đầu mang dấu `+`, các nhóm sau mang dấu `-` ở số hạng `a * b / c`, và `d` luôn mang dấu `+`).

## 5. Cách 1 — Dựng biểu thức rồi tính từng số hạng (oracle)

**Ý tưởng.** Dựng chuỗi biểu thức đúng như đề (dùng `//` cho phép chia). Tách chuỗi theo `+`/`-` thành các **số hạng** chỉ chứa `*` và `//` (ví dụ `6*5//4`); `eval` từng số hạng (an toàn vì chỉ chứa số và `*`, `//`), rồi cộng/trừ theo dấu. Mỗi số hạng chỉ chứa số **dương** nên chia sàn và chia cắt bỏ phần thập phân trùng nhau; dấu được áp dụng **sau** khi tính số hạng (đúng thứ tự ưu tiên). Không dùng ngăn xếp hay công thức, nên đây là **oracle độc lập**.

Ta không `eval` toàn bộ biểu thức vì với `n` lớn chuỗi có hàng chục nghìn ký tự và trình biên dịch Python có thể vượt giới hạn đệ quy.

In [5]:
def clumsy_eval(n: int) -> int:
    expr = build_expression(n)
    tokens = re.split(r"([+-])", expr)                 # xen kẽ: số hạng, dấu, số hạng, ...
    total = eval(tokens[0])
    for sign, term in zip(tokens[1::2], tokens[2::2]):
        value = eval(term)                             # số hạng chỉ chứa số dương, * và //
        total += value if sign == "+" else -value
    return total

for n, expected in examples:
    got = clumsy_eval(n)
    print(f"n={n}: oracle = {got}", "OK" if got == expected else "SAI")
    assert got == expected

n=4: oracle = 7 OK
n=10: oracle = 12 OK


**Độ phức tạp và nút thắt.** `O(n)` để dựng chuỗi và tính, nhưng dùng `eval` trên chuỗi (chậm, không nên dùng ở mã nộp). Đo thử:

In [6]:
for n in (100, 1000, 5000, 10000):
    t0 = time.perf_counter()
    clumsy_eval(n)
    print(f"n={n:5}: {(time.perf_counter() - t0) * 1000:8.2f} ms")

n=  100:     0.35 ms
n= 1000:     2.94 ms
n= 5000:    15.27 ms
n=10000:    29.98 ms


Nút thắt: `eval` tốn kém và không phải cách tính "thuật toán". Ta mô phỏng biểu thức bằng **ngăn xếp**.

## 6. Ý tưởng thuật toán / bước đột phá

**Mô phỏng bằng ngăn xếp, chú ý dấu với phép chia.**

- Đặt `stack = [n]`. Với mỗi số `x = n-1, n-2, ..., 1` và phép toán `op` thứ `i` (`i = 0, 1, 2, ...`, quay vòng `* / + -`):
  - `*`: `stack[-1] *= x`;
  - `/`: `stack[-1] = tdiv(stack[-1], x)` (**chia cắt bỏ phần thập phân**, để số hạng âm cho kết quả đúng như `-(a / b)`);
  - `+`: đẩy `x`;
  - `-`: đẩy `-x`.
- Đáp án là `sum(stack)`.

Cách khác không cần ngăn xếp: chia thành các **nhóm bốn số** và cộng dồn (`O(1)` bộ nhớ); và vì các nhóm "đều đặn" nên có **công thức `O(1)`** theo `n mod 4`.

| Cách | Ý tưởng |
|---|---|
| 2 | Ngăn xếp số hạng, chia cắt bỏ phần thập phân khi đỉnh âm (nộp) |
| 3 | Duyệt theo nhóm `a*b/c + d`, cộng dồn (không ngăn xếp) |
| 4 | Công thức `O(1)` theo `n mod 4` (với `n > 4`) |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Ngăn xếp (cách nộp chính)

**Giải thích từng đoạn.**

- `ops = "*/+-"`; phép toán thứ `i` là `ops[i % 4]`.
- `*`: nhân đỉnh ngăn xếp với `x`.
- `/`: chia đỉnh cho `x` **cắt bỏ phần thập phân**. Đỉnh có thể âm (số hạng sau dấu `-`), nên dùng `int(stack[-1] / x)` (chính xác trong miền này) hoặc phiên bản chỉ dùng số nguyên: `-(-a // b)` khi `a < 0`.
- `+`: đẩy `x`; `-`: đẩy `-x`.
- Trả về `sum(stack)`.

In [7]:
def clumsy_stack(n: int) -> int:
    stack = [n]
    ops = "*/+-"
    for i, x in enumerate(range(n - 1, 0, -1)):
        op = ops[i % 4]
        if op == "*":
            stack[-1] *= x
        elif op == "/":
            stack[-1] = tdiv(stack[-1], x)             # chia cắt bỏ phần thập phân (đỉnh có thể âm)
        elif op == "+":
            stack.append(x)
        else:
            stack.append(-x)
    return sum(stack)

for n, expected in examples:
    got = clumsy_stack(n)
    print(f"n={n}: ngăn xếp = {got}", "OK" if got == expected else "SAI")
    assert got == expected

n=4: ngăn xếp = 7 OK
n=10: ngăn xếp = 12 OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def clumsy(self, n: int) -> int:
        stack = [n]
        ops = "*/+-"
        for i, x in enumerate(range(n - 1, 0, -1)):
            op = ops[i % 4]
            if op == "*":
                stack[-1] *= x
            elif op == "/":
                a = stack[-1]
                stack[-1] = -((-a) // x) if a < 0 else a // x    # chia cắt bỏ phần thập phân (số hạng có thể âm)
            elif op == "+":
                stack.append(x)
            else:
                stack.append(-x)                                 # dấu trừ: đẩy số âm
        return sum(stack)

sol_stack = Solution().clumsy
for n, expected in examples:
    assert sol_stack(n) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(n)`, bộ nhớ `O(n)` (ngăn xếp chứa khoảng `n / 2` số hạng).

---

### 7.2. Cách 3 — Duyệt theo nhóm bốn số

**Ý tưởng khác.** Vì cấu trúc lặp lại, chia thành các **nhóm** `m, m-1, m-2, m-3` với biểu thức `m * (m-1) / (m-2) + (m-3)`; các nhóm nối với nhau bằng dấu `-` (nhóm đầu không có dấu `-`). Không cần ngăn xếp: cộng dồn `total`.

**Giải thích từng đoạn.**

- `m = n`, `sign = +1` (nhóm đầu mang dấu `+`).
- Tính số hạng nhân-chia của nhóm: nếu `m >= 3`: `m * (m - 1) // (m - 2)`; nếu `m == 2`: `2 * 1`; nếu `m == 1`: `1`. (Mọi số hạng đều dương nên `//` an toàn.)
- `total += sign * số_hạng`; nếu `m >= 4`: cộng thêm `(m - 3)` (số hạng `+d`, luôn dấu `+`).
- Sau nhóm đầu đặt `sign = -1`; `m -= 4`.

In [9]:
def clumsy_groups(n: int) -> int:
    total, sign, m = 0, 1, n
    while m > 0:
        if m >= 3:
            term = m * (m - 1) // (m - 2)
        elif m == 2:
            term = 2 * 1
        else:
            term = 1
        total += sign * term
        if m >= 4:
            total += m - 3
        sign = -1
        m -= 4
    return total

for n, expected in examples:
    got = clumsy_groups(n)
    print(f"n={n}: nhóm bốn số = {got}", "OK" if got == expected else "SAI")
    assert got == expected

n=4: nhóm bốn số = 7 OK
n=10: nhóm bốn số = 12 OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def clumsy(self, n: int) -> int:
        total, sign, m = 0, 1, n
        while m > 0:                           # mỗi nhóm: m * (m-1) / (m-2) + (m-3)
            if m >= 3:
                term = m * (m - 1) // (m - 2)
            elif m == 2:
                term = 2 * 1
            else:
                term = 1
            total += sign * term               # nhóm đầu dấu +, các nhóm sau dấu -
            if m >= 4:
                total += m - 3                 # số hạng "+ d" luôn dấu +
            sign = -1
            m -= 4
        return total

sol_groups = Solution().clumsy
for n, expected in examples:
    assert sol_groups(n) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(n / 4) = O(n)`, bộ nhớ `O(1)`.

---

### 7.3. Cách 4 — Công thức `O(1)` theo `n mod 4`

**Ý tưởng khác.** Với `m >= 5`, `m * (m - 1) // (m - 2) = m + 1` (vì `m(m - 1) = (m - 2)(m + 1) + 2` và `2 < m - 2`). Do đó mỗi nhóm đầy đủ sau nhóm đầu đóng góp `-(m + 1) + (m - 3) = -4`: hằng số. Cộng dồn ta được công thức theo `n mod 4`:

- `n <= 4`: `1, 2, 6, 7` (trường hợp nhỏ, tính tay);
- `n > 4` và `n mod 4 == 0`: `n + 1`;
- `n > 4` và `n mod 4` bằng `1` hoặc `2`: `n + 2`;
- `n > 4` và `n mod 4 == 3`: `n - 1`.

Chứng minh chi tiết ở mục 10; ở đây kiểm bằng máy với mọi `n` thuộc `[1, 10^4]`.

In [11]:
def clumsy_formula(n: int) -> int:
    if n <= 4:
        return [0, 1, 2, 6, 7][n]
    r = n % 4
    if r == 0:
        return n + 1
    if r <= 2:
        return n + 2
    return n - 1

for n, expected in examples:
    got = clumsy_formula(n)
    print(f"n={n}: công thức = {got}", "OK" if got == expected else "SAI")
    assert got == expected
print("dãy 1..16:", [clumsy_formula(n) for n in range(1, 17)])

n=4: công thức = 7 OK
n=10: công thức = 12 OK
dãy 1..16: [1, 2, 6, 7, 7, 8, 6, 9, 11, 12, 10, 13, 15, 16, 14, 17]


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def clumsy(self, n: int) -> int:
        if n <= 4:
            return [0, 1, 2, 6, 7][n]          # clumsy(1..4) = 1, 2, 6, 7
        r = n % 4
        if r == 0:
            return n + 1
        if r <= 2:
            return n + 2
        return n - 1

sol_formula = Solution().clumsy
for n, expected in examples:
    assert sol_formula(n) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian và bộ nhớ `O(1)`.

## 8. TRACE từng bước

### 8.1. Ngăn xếp cho `n = 10`

In từng phép toán và trạng thái ngăn xếp.

In [13]:
def trace_stack(n):
    stack = [n]
    ops = "*/+-"
    print(f"bắt đầu stack = {stack}")
    for i, x in enumerate(range(n - 1, 0, -1)):
        op = ops[i % 4]
        if op == "*":
            stack[-1] *= x
        elif op == "/":
            before = stack[-1]
            stack[-1] = tdiv(stack[-1], x)
            if before < 0:
                print(f"   (đỉnh âm {before}: chia cắt bỏ phần thập phân cho {stack[-1]}, còn // cho {before // x})")
        elif op == "+":
            stack.append(x)
        else:
            stack.append(-x)
        print(f"{op} {x:2}: stack = {stack}")
    print("tổng =", sum(stack))
    return sum(stack)

assert trace_stack(10) == 12

bắt đầu stack = [10]
*  9: stack = [90]
/  8: stack = [11]
+  7: stack = [11, 7]
-  6: stack = [11, 7, -6]
*  5: stack = [11, 7, -30]
   (đỉnh âm -30: chia cắt bỏ phần thập phân cho -7, còn // cho -8)
/  4: stack = [11, 7, -7]
+  3: stack = [11, 7, -7, 3]
-  2: stack = [11, 7, -7, 3, -2]
*  1: stack = [11, 7, -7, 3, -2]
tổng = 12


### 8.2. Nhóm bốn số cho `n = 10`

In từng nhóm và đóng góp của nó.

In [14]:
def trace_groups(n):
    total, sign, m = 0, 1, n
    while m > 0:
        if m >= 3:
            term = m * (m - 1) // (m - 2)
            desc = f"{m}*{m - 1}//{m - 2}"
        elif m == 2:
            term, desc = 2, "2*1"
        else:
            term, desc = 1, "1"
        total += sign * term
        extra = m - 3 if m >= 4 else 0
        total += extra
        print(f"nhóm m={m:2}: số hạng {desc} = {term}, dấu {sign:+d}, thêm +{extra} -> total = {total}")
        sign = -1
        m -= 4
    return total

assert trace_groups(10) == 12

nhóm m=10: số hạng 10*9//8 = 11, dấu +1, thêm +7 -> total = 18
nhóm m= 6: số hạng 6*5//4 = 7, dấu -1, thêm +3 -> total = 14
nhóm m= 2: số hạng 2*1 = 2, dấu -1, thêm +0 -> total = 12


### 8.3. Vì sao chia sàn trên số âm gây sai

Với `n = 10`: số hạng `- 6 * 5 / 4`. Nếu đẩy `-6` rồi nhân `5` được `-30`, chia sàn `-30 // 4 = -8` (sai), chia cắt bỏ phần thập phân cho `-7` (đúng). Cách sai cho `clumsy(10) = 11`, thay vì `12`.

In [15]:
def clumsy_floor_bug(n):                      # cố ý sai: dùng // trên đỉnh âm
    stack = [n]
    ops = "*/+-"
    for i, x in enumerate(range(n - 1, 0, -1)):
        op = ops[i % 4]
        if op == "*":
            stack[-1] *= x
        elif op == "/":
            stack[-1] //= x
        elif op == "+":
            stack.append(x)
        else:
            stack.append(-x)
    return sum(stack)

print("sai (dùng //):", clumsy_floor_bug(10), "| đúng:", Solution().clumsy(10))
assert clumsy_floor_bug(10) == 11 and Solution().clumsy(10) == 12

sai (dùng //): 11 | đúng: 12


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (ngăn xếp): tổng quát, dễ mở rộng sang thứ tự phép toán khác, `O(n)`.

```python
class Solution:
    def clumsy(self, n: int) -> int:
        stack = [n]
        ops = "*/+-"
        for i, x in enumerate(range(n - 1, 0, -1)):
            op = ops[i % 4]
            if op == "*":
                stack[-1] *= x
            elif op == "/":
                a = stack[-1]
                stack[-1] = -((-a) // x) if a < 0 else a // x    # chia cắt bỏ phần thập phân (số hạng có thể âm)
            elif op == "+":
                stack.append(x)
            else:
                stack.append(-x)                                 # dấu trừ: đẩy số âm
        return sum(stack)
```

In [16]:
class Solution:
    def clumsy(self, n: int) -> int:
        stack = [n]
        ops = "*/+-"
        for i, x in enumerate(range(n - 1, 0, -1)):
            op = ops[i % 4]
            if op == "*":
                stack[-1] *= x
            elif op == "/":
                a = stack[-1]
                stack[-1] = -((-a) // x) if a < 0 else a // x    # chia cắt bỏ phần thập phân (số hạng có thể âm)
            elif op == "+":
                stack.append(x)
            else:
                stack.append(-x)                                 # dấu trừ: đẩy số âm
        return sum(stack)

sol = Solution()
for n, expected in examples:
    got = sol.clumsy(n)
    print(f"n={n}: {got}", "OK" if got == expected else "SAI")
    assert got == expected

n=4: 7 OK
n=10: 12 OK


## 10. Vì sao đúng

**Thứ tự ưu tiên.** Biểu thức là tổng đại số các **số hạng**; mỗi số hạng là một chuỗi nhân/chia liên tiếp xử lý từ trái sang phải. Ngăn xếp giữ các số hạng (đã mang dấu): `+x` bắt đầu số hạng mới có dấu `+`, `-x` bắt đầu số hạng mới có dấu `-` (ta đẩy `-x` và để các phép `*`, `/` tiếp theo tác động lên số hạng **đã mang dấu**), `*` và `/` sửa **số hạng đang dở** ở đỉnh. Vì vậy `sum(stack)` bằng giá trị biểu thức.

**Chia với số hạng âm.** Số hạng "thật" là `-(a * b / c)` với `a * b / c` là chia nguyên **số dương** (nên sàn = cắt). Khi ta nhân `-a` rồi chia, ta tính `(-a * b) / c`: để bằng `-(a * b / c)`, phép chia phải **cắt bỏ phần thập phân**: `tdiv(-p, c) = -(p // c)` với `p > 0`. Chia sàn `(-p) // c` cho `-ceil(p / c)`, **sai** khi `c` không chia hết `p`.

**Cách 3.** Đúng theo phân tích nhóm: biểu thức = `[a1*b1/c1 + d1] - [a2*b2/c2] + d2 - ...`; các số hạng nhân/chia trong nhóm mang dấu `+` (nhóm đầu) hoặc `-` (nhóm sau), và `d` luôn `+`; các nhóm cuối có thể thiếu `c` hoặc `d`: `m = 3` chỉ `3 * 2 / 1`, `m = 2` chỉ `2 * 1`, `m = 1` chỉ `1`.

**Cách 4 (công thức).** Với `m >= 5`: `m(m - 1) = (m - 2)(m + 1) + 2` và `2 < m - 2`, nên `m(m - 1) // (m - 2) = m + 1`. Với `n >= 5`, nhóm đầu cho `(n + 1) + (n - 3) = 2n - 2`. Nhóm đầy đủ tiếp theo bắt đầu ở `m` (với `m >= 5`) cho `-(m + 1) + (m - 3) = -4`. Nhóm đầy đủ cuối `m = 4` cho `-(4 * 3 // 2) + 1 = -5` (vì `4 * 3 // 2 = 6 = 4 + 2`, không phải `4 + 1`). Phần dư cuối: `m = 3` cho `-(3 * 2 // 1) = -6`; `m = 2` cho `-(2 * 1) = -2`; `m = 1` cho `-1`. Cộng lại (với `g` là số nhóm đầy đủ):

- `n mod 4 = 0` (`g = n / 4 >= 2`): `2n - 2 - 4(g - 2) - 5 = n + 1`.
- `n mod 4 = 1`: các nhóm đầy đủ đều có `m >= 5`, `g = (n - 1) / 4 >= 1`, rồi `m = 1`: `2n - 2 - 4(g - 1) - 1 = n + 2`.
- `n mod 4 = 2`: tương tự với `m = 2`: `2n - 2 - 4(g - 1) - 2 = n + 2`, `g = (n - 2) / 4`.
- `n mod 4 = 3`: với `m = 3`: `2n - 2 - 4(g - 1) - 6 = n - 1`, `g = (n - 3) / 4`.

**Kiểm chứng bằng máy.** Oracle (Cách 1) dựng chuỗi biểu thức đúng cú pháp rồi tính các số hạng bằng `eval` (độc lập với ngăn xếp, nhóm và công thức). Ta so sánh công thức với cách nhóm trên **mọi** `n` từ `1` đến `10^4`, ngăn xếp và nhóm với oracle trên mọi `n <= 1500`, và lấy mẫu oracle ở phần còn lại tới `10^4`.

In [17]:
for n in range(1, 10001):                       # công thức O(1) và cách nhóm O(n/4) so với nhau trên mọi n
    assert clumsy_formula(n) == clumsy_groups(n) == sol_formula(n) == sol_groups(n), n
print("mọi n = 1..10^4: công thức và cách nhóm khớp nhau")
for n in range(1, 1501):                        # oracle eval và ngăn xếp trên mọi n nhỏ
    ref = clumsy_eval(n)
    assert clumsy_stack(n) == sol_stack(n) == sol.clumsy(n) == clumsy_groups(n) == ref, n
print("mọi n = 1..1500: ngăn xếp, nhóm khớp oracle eval")
for n in range(1501, 10001, 125):
    assert clumsy_eval(n) == clumsy_stack(n) == clumsy_formula(n), n
print("lấy mẫu n = 1501..10^4: oracle eval, ngăn xếp và công thức khớp")

mọi n = 1..10^4: công thức và cách nhóm khớp nhau


mọi n = 1..1500: ngăn xếp, nhóm khớp oracle eval


lấy mẫu n = 1501..10^4: oracle eval, ngăn xếp và công thức khớp


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Dựng chuỗi và `eval` từng số hạng | `O(n)` (chậm) | `O(n)` |
| 2. Ngăn xếp | `O(n)` | `O(n)` |
| 3. Nhóm bốn số | `O(n)` | `O(1)` |
| 4. Công thức | `O(1)` | `O(1)` |

Đo ở cỡ tối đa (`n = 10^4`), lặp nhiều lần để thấy chênh lệch:

In [18]:
n = 10 ** 4
for name, fn, reps in (("Cách 1 (oracle eval)", clumsy_eval, 20), ("Cách 2", sol_stack, 200), ("Cách 3", sol_groups, 200), ("Cách 4", sol_formula, 20000), ("nộp", sol.clumsy, 200)):
    t0 = time.perf_counter()
    for _ in range(reps):
        got = fn(n)
    dt = (time.perf_counter() - t0) / reps * 1000
    assert got == clumsy_formula(n)
    print(f"{name:22}: {dt:9.4f} ms, clumsy({n}) = {got}")

Cách 1 (oracle eval)  :   34.9623 ms, clumsy(10000) = 10001


Cách 2                :    1.2158 ms, clumsy(10000) = 10001
Cách 3                :    0.4181 ms, clumsy(10000) = 10001
Cách 4                :    0.0001 ms, clumsy(10000) = 10001


nộp                   :    1.2257 ms, clumsy(10000) = 10001


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), đối chiếu mọi cách với oracle trên **toàn miền** `n` nhỏ và lấy mẫu ở miền lớn, cùng các tính chất.

In [19]:
def all_solutions():
    return {"stack": clumsy_stack, "groups": clumsy_groups, "formula": clumsy_formula,
            "sol_stack": sol_stack, "sol_groups": sol_groups, "sol_formula": sol_formula, "final": sol.clumsy}

def check_all(n, expected=None):
    ref = clumsy_eval(n)
    for name, fn in all_solutions().items():
        assert fn(n) == ref, (name, n, fn(n), ref)
    if expected is not None:
        assert ref == expected, (n, ref, expected)
    return ref

# 12.1 ví dụ đề
for n, expected in examples:
    check_all(n, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    (1, 1),           # chỉ có "1"
    (2, 2),           # 2 * 1
    (3, 6),           # 3 * 2 / 1 = 6
    (4, 7),           # 4 * 3 / 2 + 1 = 7
    (5, 7),           # 5 * 4 / 3 + 2 - 1 = 6 + 2 - 1 = 7
    (6, 8),           # 6 * 5 / 4 + 3 - 2 * 1 = 7 + 3 - 2 = 8
    (7, 6),           # 7 * 6 / 5 + 4 - 3 * 2 / 1 = 8 + 4 - 6 = 6
    (8, 9),           # 8 * 7 / 6 + 5 - 4 * 3 / 2 + 1 = 9 + 5 - 6 + 1 = 9
    (9, 11),          # 9 * 8 / 7 + 6 - 5 * 4 / 3 + 2 - 1 = 10 + 6 - 6 + 2 - 1 = 11
    (10, 12),         # ví dụ 2
    (11, 10),         # 11 * 10 / 9 + 8 - 7 * 6 / 5 + 4 - 3 * 2 / 1 = 12 + 8 - 8 + 4 - 6 = 10
]
for n, expected in edge:
    check_all(n, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 11 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle eval)
def wrong_floor_division(n):                     # dùng // trên đỉnh ngăn xếp âm
    return clumsy_floor_bug(n)

def wrong_true_division(n):                      # dùng / (ra số thực) rồi trả về
    stack = [float(n)]
    ops = "*/+-"
    for i, x in enumerate(range(n - 1, 0, -1)):
        op = ops[i % 4]
        if op == "*":
            stack[-1] *= x
        elif op == "/":
            stack[-1] /= x
        elif op == "+":
            stack.append(x)
        else:
            stack.append(-x)
    return int(sum(stack))

def wrong_left_to_right(n):                      # bỏ qua thứ tự ưu tiên: tính tuần tự từ trái sang phải
    total = n
    ops = "*/+-"
    for i, x in enumerate(range(n - 1, 0, -1)):
        op = ops[i % 4]
        if op == "*":
            total *= x
        elif op == "/":
            total //= x
        elif op == "+":
            total += x
        else:
            total -= x
    return total

def wrong_shifted_ops(n):                        # lệch chu kỳ: dùng ops[(i + 1) % 4]
    stack = [n]
    ops = "*/+-"
    for i, x in enumerate(range(n - 1, 0, -1)):
        op = ops[(i + 1) % 4]
        if op == "*":
            stack[-1] *= x
        elif op == "/":
            stack[-1] = tdiv(stack[-1], x)
        elif op == "+":
            stack.append(x)
        else:
            stack.append(-x)
    return sum(stack)

def wrong_formula_all(n):                        # áp công thức n mod 4 cho mọi n, kể cả n <= 4
    r = n % 4
    if r == 0:
        return n + 1
    if r <= 2:
        return n + 2
    return n - 1

def find_counter(wrong, limit=40):
    for n in range(1, limit + 1):
        try:
            if wrong(n) != clumsy_eval(n):
                return n
        except Exception:
            return n
    return None

counters = {}
for name, fn in (("chia sàn trên số âm", wrong_floor_division), ("chia thực rồi ép kiểu", wrong_true_division),
                 ("tính từ trái sang phải (bỏ ưu tiên)", wrong_left_to_right), ("lệch chu kỳ phép toán", wrong_shifted_ops),
                 ("công thức cho mọi n (kể cả n <= 4)", wrong_formula_all)):
    n = find_counter(fn)
    assert n is not None, name
    counters[name] = n
    print(f"{name:38} n={n}: sai -> {fn(n)}, đúng -> {clumsy_eval(n)}")
    check_all(n)
print("12.3 phản ví dụ: OK")

chia sàn trên số âm                    n=9: sai -> 10, đúng -> 11
chia thực rồi ép kiểu                  n=9: sai -> 10, đúng -> 11
tính từ trái sang phải (bỏ ưu tiên)    n=7: sai -> 18, đúng -> 6
lệch chu kỳ phép toán                  n=3: sai -> 2, đúng -> 6
công thức cho mọi n (kể cả n <= 4)     n=1: sai -> 3, đúng -> 1
12.3 phản ví dụ: OK


In [21]:
# 12.4 toàn miền: mọi n từ 1 đến 3000 so với oracle eval; lấy mẫu n lớn
for n in range(1, 801):
    check_all(n)
rng = random.Random(4006)
for _ in range(40):
    n = rng.randint(801, 10000)
    assert clumsy_eval(n) == sol.clumsy(n) == sol_groups(n) == sol_formula(n)
assert sol.clumsy(10 ** 4) == sol_formula(10 ** 4) == sol_groups(10 ** 4)
print("12.4 mọi n = 1..800 và 40 giá trị n lớn: OK")

# 12.5 tính chất: chu kỳ 4 của hiệu clumsy(n + 4) - clumsy(n) = 4 khi n >= 5 (từ công thức), và giá trị dương
for n in range(5, 2000):
    assert sol.clumsy(n + 4) - sol.clumsy(n) == 4, n
    assert sol.clumsy(n) > 0
print("12.5 tính chất: clumsy(n + 4) - clumsy(n) = 4 với n >= 5, luôn dương: OK")

12.4 mọi n = 1..800 và 40 giá trị n lớn: OK


12.5 tính chất: clumsy(n + 4) - clumsy(n) = 4 với n >= 5, luôn dương: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chia sàn `//` trên số hạng âm

Sau dấu `-`, ta đẩy số âm và nhân; chia sàn một số âm làm tròn **xuống** (`-30 // 4 = -8`), trong khi đề muốn `-(30 / 4) = -7`.

In [22]:
n = counters["chia sàn trên số âm"]
print(f"n={n}: sai ->", wrong_floor_division(n), "| đúng ->", Solution().clumsy(n))
assert wrong_floor_division(n) != Solution().clumsy(n)
print("clumsy(10): sai", wrong_floor_division(10), "(đúng 12)")

n=9: sai -> 10 | đúng -> 11
clumsy(10): sai 11 (đúng 12)


Vì sao sai: số hạng phải là `-(a * b / c)` với chia nguyên của số dương. **Đúng:** chia **cắt bỏ phần thập phân** khi đỉnh âm (`-((-a) // x)`), hoặc tính số hạng dương rồi áp dấu.

### Lỗi 2 — Dùng `/` (số thực) rồi ép kiểu

`/` trong Python cho số thực: các chia nguyên trong chuỗi không được làm tròn đúng chỗ, và kết quả cuối có thể khác.

In [23]:
n = counters["chia thực rồi ép kiểu"]
print(f"n={n}: sai ->", wrong_true_division(n), "| đúng ->", Solution().clumsy(n))
assert wrong_true_division(n) != Solution().clumsy(n)

n=9: sai -> 10 | đúng -> 11


Vì sao sai: mỗi phép chia phải **làm tròn ngay** tại chỗ (đề: "floor division"), không giữ phần thập phân để cộng dồn rồi mới làm tròn. **Đúng:** chia nguyên ngay khi gặp `/`.

### Lỗi 3 — Bỏ qua thứ tự ưu tiên (tính từ trái sang phải)

Tính tuần tự `((10 * 9) / 8 + 7) - 6 ...` cho kết quả khác hẳn vì `*` và `/` phải được thực hiện **trước** `+` và `-`.

In [24]:
n = counters["tính từ trái sang phải (bỏ ưu tiên)"]
print(f"n={n}: sai ->", wrong_left_to_right(n), "| đúng ->", Solution().clumsy(n))
assert wrong_left_to_right(n) != Solution().clumsy(n)
print("clumsy(10): sai", wrong_left_to_right(10), "(đúng 12)")

n=7: sai -> 18 | đúng -> 6
clumsy(10): sai 16 (đúng 12)


Vì sao sai: `4 * 3 / 2 + 1` tính tuần tự vẫn ra 7 nhưng khi có `+`/`-` xen giữa chuỗi nhân/chia (`7 - 6 * 5 / 4`) thì sai. **Đúng:** dùng ngăn xếp (hoặc nhóm số hạng).

### Lỗi 4 — Lệch chu kỳ phép toán

Phép toán đầu tiên sau `n` là `*`, không phải `/`; dùng `ops[(i + 1) % 4]` hay đếm sai chỉ số làm cả biểu thức lệch.

In [25]:
n = counters["lệch chu kỳ phép toán"]
print(f"n={n}: sai ->", wrong_shifted_ops(n), "| đúng ->", Solution().clumsy(n))
assert wrong_shifted_ops(n) != Solution().clumsy(n)

n=3: sai -> 2 | đúng -> 6


Vì sao sai: với `n = 4`, biểu thức là `4 * 3 / 2 + 1`; lệch chu kỳ thành `4 / 3 + 2 - 1`. **Đúng:** phép toán thứ `i` là `"*/+-"[i % 4]` với `i` bắt đầu từ `0`.

### Lỗi 5 — Áp công thức `O(1)` cho `n <= 4`

Công thức theo `n mod 4` chỉ đúng khi `n >= 5` (vì `m(m - 1) // (m - 2) = m + 1` đòi `m >= 5`).

In [26]:
n = counters["công thức cho mọi n (kể cả n <= 4)"]
print(f"n={n}: sai ->", wrong_formula_all(n), "| đúng ->", Solution().clumsy(n))
assert wrong_formula_all(n) != Solution().clumsy(n)
print("so sánh n = 1..4:", [(k, wrong_formula_all(k), clumsy_eval(k)) for k in range(1, 5)])

n=1: sai -> 3 | đúng -> 1
so sánh n = 1..4: [(1, 3, 1), (2, 4, 2), (3, 2, 6), (4, 5, 7)]


Vì sao sai: các nhóm nhỏ (`m = 3, 4`) có tính chất khác (`4 * 3 // 2 = 6`, không phải `5`). **Đúng:** xử lý riêng `n = 1, 2, 3, 4` (`1, 2, 6, 7`).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "tính biểu thức có thứ tự ưu tiên và dấu" ⇒ **ngăn xếp số hạng**: `+`/`-` đẩy số mới (mang dấu), `*`/`/` sửa đỉnh, kết quả là tổng ngăn xếp (như bài Basic Calculator II, 227).
- **Chia số âm**: Python `//` làm tròn về `-vô cực`, C/Java cắt về `0`; đề nói "chia nguyên" thường nghĩa là **cắt về 0** khi số âm (như bài 227). Cách an toàn: tính số hạng với số **dương** rồi áp dấu.
- Khi cấu trúc lặp theo chu kỳ, thử tìm **công thức đóng** bằng cách tính vài giá trị rồi chứng minh (chu kỳ 4).
- Dùng `eval` trên các đoạn **nhỏ, an toàn** làm oracle (không dùng `eval` trên dữ liệu không tin cậy).
- Cùng họ: Basic Calculator II (227), Evaluate Reverse Polish Notation (150), Different Ways to Add Parentheses (241).
- Test nên có: `n = 1..4` (trường hợp nhỏ), `n` ứng với mọi giá trị `n mod 4`, `n = 10^4`.

## 15. Biến thể

1. **Thứ tự phép toán khác** (ví dụ `+ - * /`): ngăn xếp vẫn dùng được; công thức `O(1)` phải tính lại (bài tập 5).
2. **Chia làm tròn lên hoặc làm tròn gần nhất.**
3. **Tính với số `n` rất lớn** (ví dụ `10^18`): dùng công thức `O(1)`.
4. Bài liên quan: 227, 150, 241.

Biến thể 1 (thứ tự phép toán tùy ý, quay vòng) kiểm bằng oracle dựng chuỗi:

In [27]:
def clumsy_general(n: int, rotation: str) -> int:
    """Ngăn xếp với thứ tự phép toán quay vòng tùy ý (hoán vị của '*/+-')."""
    stack = [n]
    for i, x in enumerate(range(n - 1, 0, -1)):
        op = rotation[i % len(rotation)]
        if op == "*":
            stack[-1] *= x
        elif op == "/":
            stack[-1] = tdiv(stack[-1], x)
        elif op == "+":
            stack.append(x)
        else:
            stack.append(-x)
    return sum(stack)

def clumsy_general_oracle(n: int, rotation: str) -> int:
    py = {"*": "*", "/": "//", "+": "+", "-": "-"}
    parts = [str(n)]
    for i, x in enumerate(range(n - 1, 0, -1)):
        parts += [py[rotation[i % len(rotation)]], str(x)]
    tokens = re.split(r"([+-])", "".join(parts))
    total = eval(tokens[0])
    for sign, term in zip(tokens[1::2], tokens[2::2]):
        total += eval(term) if sign == "+" else -eval(term)
    return total

assert clumsy_general(10, "*/+-") == 12
rng = random.Random(15)
import itertools
for rotation in ("".join(p) for p in itertools.permutations("*/+-")):
    for n in range(1, 60):
        assert clumsy_general(n, rotation) == clumsy_general_oracle(n, rotation), (rotation, n)
print("biến thể quay vòng tùy ý (24 hoán vị của '*/+-', n = 1..59): ngăn xếp khớp oracle; ví dụ '+-*/' với n = 10:", clumsy_general(10, "+-*/"))

biến thể quay vòng tùy ý (24 hoán vị của '*/+-', n = 1..59): ngăn xếp khớp oracle; ví dụ '+-*/' với n = 10: 10


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Dựng chuỗi + `eval` số hạng | Theo đề | `O(n)` | `O(n)` | Oracle |
| 2. Ngăn xếp | Số hạng có dấu, chia cắt bỏ phần thập phân | `O(n)` | `O(n)` | **Cách nộp chính**, tổng quát |
| 3. Nhóm bốn số | Cộng dồn từng nhóm | `O(n)` | `O(1)` | Tiết kiệm bộ nhớ |
| 4. Công thức `n mod 4` | Chu kỳ 4 | `O(1)` | `O(1)` | `n` rất lớn |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(10 * 9 // 8, -6 * 5 // 4, -(6 * 5 // 4), tdiv(-30, 4))
print(Solution().clumsy(1), Solution().clumsy(4), Solution().clumsy(5))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # bốn số
my_guess_2 = None      # ba số

### Bài 2 — Tính tay (dễ-vừa)

Tính tay `clumsy(7) = 7 * 6 / 5 + 4 - 3 * 2 / 1` (nhớ chia nguyên).

In [29]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành ngăn xếp.

In [30]:
def exercise_3(n: int) -> int:
    stack = [n]
    ops = "*/+-"
    for i, x in enumerate(range(n - 1, 0, -1)):
        op = ops[i % 4]
        # TODO 1: "*" -> nhân đỉnh ngăn xếp với x
        # TODO 2: "/" -> chia đỉnh cho x, CẮT BỎ phần thập phân (đỉnh có thể âm)
        # TODO 3: "+" -> đẩy x; "-" -> đẩy -x
        pass
    return sum(stack)


# for n, expected in examples:
#     assert exercise_3(n) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, tự viết `exercise_4(n)` bằng **duyệt theo nhóm bốn số** (cộng dồn, `O(1)` bộ nhớ).

In [31]:
def exercise_4(n: int) -> int:
    # TODO
    pass


# for n, expected in examples:
#     assert exercise_4(n) == expected

### Bài 5 — Mini challenge: thứ tự phép toán tùy ý (khó)

Viết `exercise_5(n, rotation)` trả về giá trị biểu thức `n op1 (n-1) op2 (n-2) ...` trong đó phép toán thứ `i` là `rotation[i % 4]` (với `rotation` là một hoán vị của `"*/+-"`), cùng quy tắc ưu tiên và chia nguyên (cắt bỏ phần thập phân) như đề.

```python
exercise_5(10, "*/+-") -> 12     # chính là clumsy(10)
exercise_5(4, "+-*/")  -> 5      # 4 + 3 - 2 * 1 = 5
exercise_5(5, "-*/+")  -> 0      # 5 - 4 * 3 / 2 + 1 = 5 - 6 + 1 = 0
exercise_5(1, "*/+-")  -> 1
```
Gợi ý: ngăn xếp như Cách 2 nhưng chọn phép toán theo `rotation`.

In [32]:
def exercise_5(n: int, rotation: str) -> int:
    # TODO
    pass


# assert exercise_5(10, "*/+-") == 12
# assert exercise_5(4, "+-*/") == 5
# assert exercise_5(5, "-*/+") == 0
# assert exercise_5(1, "*/+-") == 1

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `//` là chia sàn (làm tròn về `-vô cực`); `tdiv` cắt về `0`.
- Mức 2: `-6 * 5 // 4 = -30 // 4 = -8`; `-(6 * 5 // 4) = -(7) = -7`.
- Mức 3: dòng 1: `11 -8 -7 -7`; dòng 2: `1 7 7`.

**Bài 2**
- Mức 1: số hạng 1: `7 * 6 / 5 = 42 / 5 = 8`.
- Mức 2: `+ 4`; số hạng cuối: `- (3 * 2 / 1) = - 6`.
- Mức 3: `8 + 4 - 6 = 6`.

**Bài 3**
- Mức 1: `if op == "*": stack[-1] *= x`.
- Mức 2: `elif op == "/": a = stack[-1]; stack[-1] = -((-a) // x) if a < 0 else a // x`.
- Mức 3: `elif op == "+": stack.append(x)` và `else: stack.append(-x)`.

**Bài 4**
- Mức 1: nhóm `m, m-1, m-2, m-3` với số hạng `m*(m-1)//(m-2)` rồi `+ (m-3)`.
- Mức 2: nhóm đầu dấu `+`, các nhóm sau dấu `-`; `m = 3, 2, 1` là nhóm cuối đặc biệt.
- Mức 3: dùng `sign`; `m -= 4` sau mỗi nhóm.

**Bài 5**
- Mức 1: giống Cách 2, thay `ops = rotation`.
- Mức 2: `op = rotation[i % 4]`.
- Mức 3: ví dụ `exercise_5(5, "-*/+")`: `5 - 4 * 3 / 2 + 1`: đẩy `-4`, nhân `3` được `-12`, chia `2` (cắt bỏ phần thập phân) được `-6`, đẩy `1`: `5 - 6 + 1 = 0`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
print(10 * 9 // 8, -6 * 5 // 4, -(6 * 5 // 4), tdiv(-30, 4))
print(Solution().clumsy(1), Solution().clumsy(4), Solution().clumsy(5))

assert (10 * 9 // 8, -6 * 5 // 4, -(6 * 5 // 4), tdiv(-30, 4)) == (11, -8, -7, -7)
assert (Solution().clumsy(1), Solution().clumsy(4), Solution().clumsy(5)) == (1, 7, 7)

11 -8 -7 -7
1 7 7


### Đáp án Bài 2

`7 * 6 / 5 = 42 // 5 = 8`; `+ 4`; `- (3 * 2 / 1) = - 6`. Tổng `8 + 4 - 6 = 6`.

In [34]:
hand_answer_solution = 6
assert Solution().clumsy(7) == hand_answer_solution == clumsy_eval(7)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_solution(n: int) -> int:
    stack = [n]
    ops = "*/+-"
    for i, x in enumerate(range(n - 1, 0, -1)):
        op = ops[i % 4]
        if op == "*":
            stack[-1] *= x
        elif op == "/":
            a = stack[-1]
            stack[-1] = -((-a) // x) if a < 0 else a // x
        elif op == "+":
            stack.append(x)
        else:
            stack.append(-x)
    return sum(stack)

for n, expected in examples:
    assert exercise_3_solution(n) == expected
for n in range(1, 500):
    assert exercise_3_solution(n) == clumsy_eval(n)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_solution(n: int) -> int:
    total, sign, m = 0, 1, n
    while m > 0:
        if m >= 3:
            term = m * (m - 1) // (m - 2)
        elif m == 2:
            term = 2
        else:
            term = 1
        total += sign * term
        if m >= 4:
            total += m - 3
        sign = -1
        m -= 4
    return total

for n, expected in examples:
    assert exercise_4_solution(n) == expected
for n in range(1, 500):
    assert exercise_4_solution(n) == clumsy_eval(n)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_solution(n: int, rotation: str) -> int:
    stack = [n]
    for i, x in enumerate(range(n - 1, 0, -1)):
        op = rotation[i % 4]
        if op == "*":
            stack[-1] *= x
        elif op == "/":
            a = stack[-1]
            stack[-1] = -((-a) // x) if a < 0 else a // x
        elif op == "+":
            stack.append(x)
        else:
            stack.append(-x)
    return sum(stack)

assert exercise_5_solution(10, "*/+-") == 12
assert exercise_5_solution(4, "+-*/") == 5
assert exercise_5_solution(5, "-*/+") == 0
assert exercise_5_solution(1, "*/+-") == 1
for rotation in ("".join(p) for p in itertools.permutations("*/+-")):
    for n in range(1, 40):
        assert exercise_5_solution(n, rotation) == clumsy_general_oracle(n, rotation), (rotation, n)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (ngăn xếp số hạng)

```python
class Solution:
    def clumsy(self, n: int) -> int:
        stack = [n]
        ops = "*/+-"
        for i, x in enumerate(range(n - 1, 0, -1)):
            op = ops[i % 4]
            if op == "*":
                stack[-1] *= x
            elif op == "/":
                a = stack[-1]
                stack[-1] = -((-a) // x) if a < 0 else a // x    # chia cắt bỏ phần thập phân
            elif op == "+":
                stack.append(x)
            else:
                stack.append(-x)                                 # dấu trừ: đẩy số âm
        return sum(stack)
```

Công thức nhớ nhanh: `clumsy(n)` với `n >= 5`: `n + 1` nếu `n mod 4 = 0`, `n + 2` nếu `n mod 4` là `1` hoặc `2`, `n - 1` nếu `n mod 4 = 3`; còn `clumsy(1..4) = 1, 2, 6, 7`.

### Checklist trước khi submit

- [ ] Phép toán thứ `i` là `"*/+-"[i % 4]` với `i` bắt đầu từ `0` (phép đầu tiên là `*`)?
- [ ] `*` và `/` tác động lên **số hạng đang dở** (đỉnh ngăn xếp), `+`/`-` đẩy số mới (mang dấu)?
- [ ] Chia **cắt bỏ phần thập phân** khi đỉnh âm (không dùng `//` trần trên số âm)?
- [ ] Không dùng `/` cho ra số thực; kết quả trả về là `int`?
- [ ] Nếu dùng công thức `O(1)`: xử lý riêng `n <= 4`?
- [ ] Đã thử: `n = 1, 2, 3, 4, 5, 10`, mọi giá trị `n mod 4`, `n = 10^4`?